**IMPROVEMENTS**

* [] pass the json for validation directly to the model invocation when possible (check if it's supported)

Self healing strategies:

* [x] increase temperature error after error (already implemented, works sometimes for qwen3)
* [ ] pass the error message when problems in the json are detected to see if it fixes them afterwards (new prompt with schema, invalid JSON, validation error and "Fix the JSON so it passes validation. Do not change values unless required to fix an error.")

In [1]:
import sys
import time
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os

from prompts import SYSTEM_PROMPT
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction

In [2]:
os.chdir("..")

In [3]:
LANGUAGE = "en" #TODO
# MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
MODEL_ID = "openai/gpt-oss-120b"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_dynamic-temp_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

FS_DATASET = "NLP-FBK/dyspnea-crf-train"
DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-development"
NUM_FS_EXAMPLES = 3
MAX_RETRIES = 3

API_KEY = "REDACTED_API_KEY" #TODO move
if not API_KEY: #TODO remove or make it useful
    print("Error: GROQ_API_KEY environment variable not set.")
    print("Please export your API key: export GROQ_API_KEY='your_key_here'")
    raise ValueError

In [4]:
def create_base_messages(model: type[BaseModel]):
    # system prompt and expected schema
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + "\n\nCRF Schema:\n" + build_compact_schema(model)
        }
    ]

    # few-shot examples
    fs_examples = load_or_download_dataset(FS_DATASET)[LANGUAGE]
    for ex in fs_examples.select(range(NUM_FS_EXAMPLES)):
        messages.append({
            "role": "user",
            "content": f"Extract structured data from this clinical note:\n\n{ex["clinical_note"]}"
        })
        # Filter out unknown values to save tokens and match system prompt instructions
        filtered_annotations = {item["item"]: item["ground_truth"] for item in ex["annotations"] if item["ground_truth"] != "unknown"}
        messages.append({
            "role": "assistant",
            "content": json.dumps(filtered_annotations, ensure_ascii=False)
        })
    return messages

In [5]:
def invoke_model(client, clinical_note: str, messages, attempt: int):
    messages.append({
        "role": "user",
        "content": f"""
Now process the following clinical note: 
{clinical_note}
"""
    })

    temperature = attempt * 0.2 # NOTE first shy attempt to make the model escape the same error over and over

    # Let exceptions propagate to be handled by the caller
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=temperature,# default = 1
        reasoning_format="hidden" if MODEL_ID == "qwen/qwen3-32b" else None
    )

    raw_output = response.choices[0].message.content
    return raw_output

In [6]:
# one-time initializations
client = Groq(api_key=API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

base_messages = create_base_messages(CRFModel)
base_messages

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-development, loading from disk...
NLP-FBK/dyspnea-crf-development has 80 documents
Found local cache at data/dyspnea-crf-train, loading from disk...


[{'role': 'system',
  'content': "\nYou extract structured clinical data from a single clinical note.\n\nTask:\nPopulate a Dyspnea Case Report Form (CRF) using ONLY information explicitly stated in the note.\n\nRules:\n- Extract data strictly from the text; do not infer, guess, or normalize.\n- Populate a field ONLY if there is clear, explicit evidence.\n- If information is absent, unclear, ambiguous, or contradictory, treat it as unknown.\n- OMIT fields with unknown values from the output JSON.\n- All fields are strings.\n- For each populated field, output EXACTLY one of the allowed values defined in the schema.\n- Do not add, remove, rename fields, or invent values.\n- Output MUST be valid, minified JSON only (no whitespace, no explanations, no extra text).\n\n\nCRF Schema:\nchronic pulmonary disease: certainly chronic | possibly chronic | certainly not chronic | unknown\nchronic respiratory failure: certainly chronic | possibly chronic | certainly not chronic | unknown\nchronic card

In [10]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

(57,
 {'1014081_en',
  '1015214_en',
  '1091594_en',
  '1117595_en',
  '114284_en',
  '1147182_en',
  '1166267_en',
  '1175847_en',
  '1195735_en',
  '1217595_en',
  '1222948_en',
  '1231372_en',
  '1236585_en',
  '1258026_en',
  '1262429_en',
  '1307609_en',
  '1323348_en',
  '1323973_en',
  '1350262_en',
  '1486650_en',
  '1498690_en',
  '1526466_en',
  '15455_en',
  '1565152_en',
  '1706077_en',
  '1769015_en',
  '1777967_en',
  '1803820_en',
  '1833659_en',
  '243838_en',
  '333752_en',
  '382962_en',
  '419929_en',
  '530889_en',
  '548625_en',
  '572916_en',
  '597775_en',
  '606486_en',
  '614605_en',
  '627570_en',
  '63875_en',
  '643686_en',
  '646228_en',
  '646639_en',
  '667544_en',
  '681196_en',
  '709578_en',
  '769230_en',
  '772892_en',
  '785051_en',
  '787885_en',
  '821011_en',
  '854863_en',
  '893897_en',
  '94439_en',
  '947751_en',
  '987763_en'})

In [ ]:
#TODO clean this functiton
def crf_dataset():
    with open(OUTPUT_PATH, "a") as f: #TODO this is append now
        for item in ds:
            doc_id = item["document_id"]
            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            clinical_note = item["clinical_note"]
            
            print(f"Processing document: {doc_id}")
            success = False

            for attempt in range(MAX_RETRIES):
                try:
                    # 1. Invoke model (Network/API errors cached here)
                    raw_output = invoke_model(client, clinical_note, base_messages.copy(), attempt)
                    
                    # 2. Parse JSON
                    data_dict = json.loads(raw_output)
                    
                    # 3. Validate with Pydantic
                    result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success

                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except Exception as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts.")
                
                # Fallback: Create fake instance with all "unknown"
                fake_instance = CRFModel()
                save_prediction(f, doc_id, fake_instance)

Skipped 1014081_en because already processed
Skipped 1117595_en because already processed
Skipped 114284_en because already processed
Skipped 1147182_en because already processed
Skipped 1166267_en because already processed
Skipped 1231372_en because already processed
Skipped 1236585_en because already processed
Skipped 1258026_en because already processed
Skipped 1307609_en because already processed
Skipped 1323348_en because already processed
Skipped 1323973_en because already processed
Skipped 1350262_en because already processed
Skipped 1486650_en because already processed
Skipped 1498690_en because already processed
Skipped 1526466_en because already processed
Skipped 15455_en because already processed
Skipped 1565152_en because already processed
Skipped 1706077_en because already processed
Skipped 1769015_en because already processed
Skipped 1777967_en because already processed
Skipped 1803820_en because already processed
Skipped 1833659_en because already processed
Skipped 24383

KeyboardInterrupt: 